In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from itertools import product

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import (
    RandomForestClassifier, 
    ExtraTreesClassifier, 
    GradientBoostingClassifier, 
    AdaBoostClassifier
)

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.utils import resample
import time
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.metrics import classification_report, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
df = pd.read_csv('base-limpia.csv')

In [3]:
# Separar target de features
target = 'fraud_bool'
X = df.drop(columns=[target])
y = df[target]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [6]:


print("Iniciando pipeline de entrenamiento y evaluación...")
print("-" * 60)

# =====================================================================
# 0. CREAR SUBCONJUNTO ESTRATIFICADO (10% de Train para GridSearch)
# =====================================================================
# Esto mantiene la proporción exacta de fraudes (0 y 1) pero usa menos datos para ser rápido
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.90, random_state=42)
for train_idx_small, _ in sss.split(X_train, y_train):
    X_train_small = X_train.iloc[train_idx_small]
    y_train_small = y_train.iloc[train_idx_small]

print(f"Tamaño Train Original: {X_train.shape[0]} muestras.")
print(f"Tamaño Train Submuestreado (10%): {X_train_small.shape[0]} muestras para GridSearch.\n")

# =====================================================================
# 1. SVM LINEAL (Optimizado para datasets masivos)
# =====================================================================
print("Entrenando LinearSVC...")
start = time.time()
pipe_svc = Pipeline([
    ('scaler', StandardScaler()), 
    ('svm', LinearSVC(random_state=42, class_weight='balanced', dual='auto', max_iter=5000))
])

param_svc = {'svm__C': [0.01, 0.1, 1, 10]}
grid_svc = GridSearchCV(pipe_svc, param_svc, cv=5, scoring='average_precision', n_jobs=-1)
grid_svc.fit(X_train_small, y_train_small)

print(f"Mejor SVM: {grid_svc.best_params_}")
print(f"AP Score (CV): {grid_svc.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 2. ÁRBOL DE DECISIÓN (Con Poda)
# =====================================================================
print("Entrenando Decision Tree...")
start = time.time()
tree = DecisionTreeClassifier(random_state=42, class_weight='balanced')

param_tree = {
    'max_depth': [5, 10, 20, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'ccp_alpha': [0.0, 0.001, 0.01]
}
grid_tree = GridSearchCV(tree, param_tree, cv=5, scoring='average_precision', n_jobs=-1)
grid_tree.fit(X_train_small, y_train_small)

print(f"Mejor Árbol: {grid_tree.best_params_}")
print(f"AP Score (CV): {grid_tree.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 3. RANDOM FOREST
# =====================================================================
print("Entrenando Random Forest...")
start = time.time()
rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)

param_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 5]
}
grid_rf = GridSearchCV(rf, param_rf, cv=5, scoring='average_precision', n_jobs=-1)
grid_rf.fit(X_train_small, y_train_small)

print(f"Mejor RF: {grid_rf.best_params_}")
print(f"AP Score (CV): {grid_rf.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 4. GRADIENT BOOSTING (Con manejo de pesos manual)
# =====================================================================
print("Entrenando Gradient Boosting...")
start = time.time()
gb = GradientBoostingClassifier(random_state=42)

param_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

# Calculamos los pesos para pasarlos al GB, ya que no tiene class_weight nativo
pesos_train_small = compute_sample_weight(class_weight='balanced', y=y_train_small)

grid_gb = GridSearchCV(gb, param_gb, cv=5, scoring='average_precision', n_jobs=-1)
# Pasamos los pesos a través del parámetro fit_params
grid_gb.fit(X_train_small, y_train_small, sample_weight=pesos_train_small)

print(f"Mejor GB: {grid_gb.best_params_}")
print(f"AP Score (CV): {grid_gb.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 5. EVALUACIÓN FINAL: ENTRENAR EL MEJOR MODELO CON EL 100% DE LOS DATOS
# =====================================================================
print("-" * 60)
print("SELECCIONANDO EL MEJOR MODELO GENERAL...")

# Comparamos el score AP de todos los grids para encontrar al ganador absoluto
resultados = {
    "LinearSVC": (grid_svc.best_score_, grid_svc.best_estimator_),
    "DecisionTree": (grid_tree.best_score_, grid_tree.best_estimator_),
    "RandomForest": (grid_rf.best_score_, grid_rf.best_estimator_),
    "GradientBoosting": (grid_gb.best_score_, grid_gb.best_estimator_)
}

# Encontrar el modelo con el mejor AP Score
nombre_mejor_modelo = max(resultados, key=lambda k: resultados[k][0])
mejor_modelo_final = resultados[nombre_mejor_modelo][1]

print(f"¡El modelo ganador es {nombre_mejor_modelo} con un AP Score en CV de {resultados[nombre_mejor_modelo][0]:.4f}!")
print("Reentrenando el modelo ganador con el 100% de los datos de entrenamiento...")

# Reentrenamos con el 100% de Train (incluyendo manejo de pesos si es GB)
start_final = time.time()
if nombre_mejor_modelo == "GradientBoosting":
    pesos_train_full = compute_sample_weight(class_weight='balanced', y=y_train)
    mejor_modelo_final.fit(X_train, y_train, sample_weight=pesos_train_full)
else:
    mejor_modelo_final.fit(X_train, y_train)
    
print(f"Entrenamiento final completado en {time.time()-start_final:.2f}s.\n")

# =====================================================================
# 6. MÉTRICAS EN TEST
# =====================================================================
print(f"--- REPORTE DE CLASIFICACIÓN EN TEST ({nombre_mejor_modelo}) ---")

# Predecir clases e inferir probabilidades
y_pred = mejor_modelo_final.predict(X_test)

# Excepción para LinearSVC que usa decision_function en lugar de predict_proba
if nombre_mejor_modelo == "LinearSVC":
    y_proba = mejor_modelo_final.decision_function(X_test)
else:
    y_proba = mejor_modelo_final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
ap_test = average_precision_score(y_test, y_proba)
print("-" * 60)
print(f"Average Precision (AP) Final en Test: {ap_test:.4f}")
print("-" * 60)

Iniciando pipeline de entrenamiento y evaluación...
------------------------------------------------------------
Tamaño Train Original: 800000 muestras.
Tamaño Train Submuestreado (10%): 80000 muestras para GridSearch.

Entrenando LinearSVC...
Mejor SVM: {'svm__C': 0.01}
AP Score (CV): 0.1368 | Tiempo: 26.42s

Entrenando Decision Tree...
Mejor Árbol: {'ccp_alpha': 0.001, 'max_depth': 20, 'min_samples_leaf': 20}
AP Score (CV): 0.0480 | Tiempo: 40.25s

Entrenando Random Forest...
Mejor RF: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}
AP Score (CV): 0.1255 | Tiempo: 102.93s

Entrenando Gradient Boosting...
Mejor GB: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
AP Score (CV): 0.8724 | Tiempo: 282.97s

------------------------------------------------------------
SELECCIONANDO EL MEJOR MODELO GENERAL...
¡El modelo ganador es GradientBoosting con un AP Score en CV de 0.8724!
Reentrenando el modelo ganador con el 100% de los datos de entrenamiento...


KeyboardInterrupt: 

In [8]:
# =====================================================================
# EXTRA. ADABOOST
# =====================================================================
print("Entrenando AdaBoost...")
start = time.time()

# Para que AdaBoost entienda el desbalance, le pasamos un árbol base "balanceado"
base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42, class_weight='balanced')

ada = AdaBoostClassifier(estimator=base_estimator, random_state=42)

param_ada = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.5, 1.0]
}

grid_ada = GridSearchCV(ada, param_ada, cv=10, scoring='average_precision', n_jobs=-1)
grid_ada.fit(X_train_small, y_train_small)

print(f"Mejor AdaBoost: {grid_ada.best_params_}")
print(f"AP Score (CV): {grid_ada.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

Entrenando AdaBoost...
Mejor AdaBoost: {'learning_rate': 0.1, 'n_estimators': 50}
AP Score (CV): 0.0193 | Tiempo: 11.12s



In [9]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.metrics import classification_report, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

print("Iniciando pipeline de entrenamiento y evaluación...")
print("-" * 60)

# =====================================================================
# 0. CREAR SUBCONJUNTO ESTRATIFICADO (10% de Train para GridSearch)
# =====================================================================
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.90, random_state=42)
for train_idx_small, _ in sss.split(X_train, y_train):
    X_train_small = X_train.iloc[train_idx_small]
    y_train_small = y_train.iloc[train_idx_small]

print(f"Tamaño Train Original: {X_train.shape[0]} muestras.")
print(f"Tamaño Train Submuestreado (10%): {X_train_small.shape[0]} muestras para GridSearch.\n")

# =====================================================================
# 1. SVM LINEAL
# =====================================================================
print("Entrenando LinearSVC...")
start = time.time()
pipe_svc = Pipeline([
    ('scaler', StandardScaler()), 
    ('svm', LinearSVC(random_state=42, class_weight='balanced', dual='auto', max_iter=5000))
])

param_svc = {'svm__C': [0.01, 0.1, 1, 10]}
grid_svc = GridSearchCV(pipe_svc, param_svc, cv=5, scoring='average_precision', n_jobs=-1)
grid_svc.fit(X_train_small, y_train_small)

print(f"Mejor SVM: {grid_svc.best_params_}")
print(f"AP Score (CV): {grid_svc.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 2. ÁRBOL DE DECISIÓN
# =====================================================================
print("Entrenando Decision Tree...")
start = time.time()
tree = DecisionTreeClassifier(random_state=42, class_weight='balanced')

param_tree = {
    'max_depth': [5, 10, 20, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'ccp_alpha': [0.0, 0.001, 0.01]
}
grid_tree = GridSearchCV(tree, param_tree, cv=5, scoring='average_precision', n_jobs=-1)
grid_tree.fit(X_train_small, y_train_small)

print(f"Mejor Árbol: {grid_tree.best_params_}")
print(f"AP Score (CV): {grid_tree.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 3. RANDOM FOREST
# =====================================================================
print("Entrenando Random Forest...")
start = time.time()
rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)

param_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 5]
}
grid_rf = GridSearchCV(rf, param_rf, cv=5, scoring='average_precision', n_jobs=-1)
grid_rf.fit(X_train_small, y_train_small)

print(f"Mejor RF: {grid_rf.best_params_}")
print(f"AP Score (CV): {grid_rf.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 4. GRADIENT BOOSTING
# =====================================================================
print("Entrenando Gradient Boosting...")
start = time.time()
gb = GradientBoostingClassifier(random_state=42)

param_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

pesos_train_small = compute_sample_weight(class_weight='balanced', y=y_train_small)
grid_gb = GridSearchCV(gb, param_gb, cv=5, scoring='average_precision', n_jobs=-1)
grid_gb.fit(X_train_small, y_train_small, sample_weight=pesos_train_small)

print(f"Mejor GB: {grid_gb.best_params_}")
print(f"AP Score (CV): {grid_gb.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 4.5 ADABOOST
# =====================================================================
print("Entrenando AdaBoost...")
start = time.time()
base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42, class_weight='balanced')
ada = AdaBoostClassifier(estimator=base_estimator, random_state=42)

param_ada = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.5, 1.0]
}

grid_ada = GridSearchCV(ada, param_ada, cv=5, scoring='average_precision', n_jobs=-1)
grid_ada.fit(X_train_small, y_train_small)

print(f"Mejor AdaBoost: {grid_ada.best_params_}")
print(f"AP Score (CV): {grid_ada.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 5. SELECCIÓN DEL MEJOR MODELO Y REENTRENAMIENTO (100% DATOS)
# =====================================================================
print("-" * 60)
print("SELECCIONANDO EL MEJOR MODELO GENERAL...")

resultados = {
    "LinearSVC": (grid_svc.best_score_, grid_svc.best_estimator_),
    "DecisionTree": (grid_tree.best_score_, grid_tree.best_estimator_),
    "RandomForest": (grid_rf.best_score_, grid_rf.best_estimator_),
    "GradientBoosting": (grid_gb.best_score_, grid_gb.best_estimator_),
    "AdaBoost": (grid_ada.best_score_, grid_ada.best_estimator_) # Añadido AdaBoost aquí
}

nombre_mejor_modelo = max(resultados, key=lambda k: resultados[k][0])
mejor_modelo_final = resultados[nombre_mejor_modelo][1]

print(f"¡El modelo ganador es {nombre_mejor_modelo} con un AP Score en CV de {resultados[nombre_mejor_modelo][0]:.4f}!")
print("Reentrenando el modelo ganador con el 100% de los datos de entrenamiento...")

start_final = time.time()
if nombre_mejor_modelo == "GradientBoosting":
    pesos_train_full = compute_sample_weight(class_weight='balanced', y=y_train)
    mejor_modelo_final.fit(X_train, y_train, sample_weight=pesos_train_full)
else:
    mejor_modelo_final.fit(X_train, y_train)
    
print(f"Entrenamiento final completado en {time.time()-start_final:.2f}s.\n")

# =====================================================================
# 6. MÉTRICAS EN TEST
# =====================================================================
print(f"--- REPORTE DE CLASIFICACIÓN EN TEST ({nombre_mejor_modelo}) ---")

y_pred = mejor_modelo_final.predict(X_test)

if nombre_mejor_modelo == "LinearSVC":
    y_proba = mejor_modelo_final.named_steps['svm'].decision_function(X_test)
else:
    y_proba = mejor_modelo_final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
ap_test = average_precision_score(y_test, y_proba)
print("-" * 60)
print(f"Average Precision (AP) Final en Test: {ap_test:.4f}")
print("-" * 60)

# =====================================================================
# 7. EXTRA: GRÁFICO DE IMPORTANCIA DE VARIABLES (FEATURE IMPORTANCE)
# =====================================================================
# Este bloque enriquecerá muchísimo tu EDA y conclusiones
print("Generando gráfico de Importancia de Variables...")

# Comprobamos si el modelo está en un Pipeline (como LinearSVC) o es un modelo directo
modelo_para_importancias = mejor_modelo_final.named_steps['svm'] if nombre_mejor_modelo == "LinearSVC" else mejor_modelo_final

if hasattr(modelo_para_importancias, 'feature_importances_'):
    importancias = modelo_para_importancias.feature_importances_
elif hasattr(modelo_para_importancias, 'coef_'):
    importancias = np.abs(modelo_para_importancias.coef_[0]) # Tomamos el valor absoluto para SVM Lineal
else:
    importancias = None

if importancias is not None:
    df_importancias = pd.DataFrame({
        'Variable': X_train.columns,
        'Importancia': importancias
    }).sort_values(by='Importancia', ascending=False).head(15) # Top 15 variables
    
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importancia', y='Variable', data=df_importancias, palette='viridis')
    plt.title(f'Top 15 Variables más Importantes - {nombre_mejor_modelo}')
    plt.xlabel('Importancia (Magnitud)')
    plt.ylabel('Variable')
    plt.tight_layout()
    plt.show()
else:
    print("El modelo seleccionado no soporta la extracción directa de importancia de variables.")

Iniciando pipeline de entrenamiento y evaluación...
------------------------------------------------------------
Tamaño Train Original: 800000 muestras.
Tamaño Train Submuestreado (10%): 80000 muestras para GridSearch.

Entrenando LinearSVC...
Mejor SVM: {'svm__C': 0.01}
AP Score (CV): 0.1368 | Tiempo: 10.92s

Entrenando Decision Tree...
Mejor Árbol: {'ccp_alpha': 0.001, 'max_depth': 20, 'min_samples_leaf': 20}
AP Score (CV): 0.0480 | Tiempo: 35.13s

Entrenando Random Forest...
Mejor RF: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}
AP Score (CV): 0.1255 | Tiempo: 90.95s

Entrenando Gradient Boosting...
Mejor GB: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
AP Score (CV): 0.8724 | Tiempo: 279.55s

Entrenando AdaBoost...
Mejor AdaBoost: {'learning_rate': 0.1, 'n_estimators': 50}
AP Score (CV): 0.0192 | Tiempo: 6.10s

------------------------------------------------------------
SELECCIONANDO EL MEJOR MODELO GENERAL...
¡El modelo ganador es GradientBoostin

KeyboardInterrupt: 